# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krashishkr008-ghg/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis + Time Window

- **One row represents:** One pseudonymized content item, identified by `content_id` and associated with one pseudonymized `client_id`.
- **Performance window:** The traffic and engagement totals cover the trailing 90 days; the comparison fields cover the last 30 days versus the previous 30 days.
- **Date limitation:** The starter CSV has no calendar date or month column, so this contract cannot claim a calendar-period window.
- **Decision supported:** Use observed page-level signals to rank content for review; this is decision-support, not a causal claim.

In [4]:
from pathlib import Path
from urllib.request import urlopen
from io import BytesIO
import pandas as pd

DATA_CANDIDATES = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
local_path = next((path for path in DATA_CANDIDATES if path.exists()), None)
if local_path is not None:
    DATA_PATH = str(local_path)
    df = pd.read_csv(local_path)
else:
    DATA_URL = "https://raw.githubusercontent.com/somnathsutra/ML-01-ASSIGNMENT/main/data/raw/content_refresh_anonymized.csv"
    with urlopen(DATA_URL) as response:
        df = pd.read_csv(BytesIO(response.read()))
    DATA_PATH = DATA_URL

print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns from {DATA_PATH}")
print("Columns:", list(df.columns))

Loaded 30,000 rows and 44 columns from https://raw.githubusercontent.com/somnathsutra/ML-01-ASSIGNMENT/main/data/raw/content_refresh_anonymized.csv
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields

### Features
These are observed before a review decision and may be used as candidate inputs:
- `search_volume`, `competition`, `cpc`
- `word_count`, `char_count`, `content_age_days`, `days_since_last_update`
- `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`
- `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`
- `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`
- `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`
- categorical descriptors such as `competition_level`, `content_type`, `main_intent`, and the age, freshness, length, impression, and position tiers

### Label / proxy
- `is_declining_label = (trend_direction == "down")`

This is the starter pipeline's binary proxy for observed decline. It is computed from `trend_direction` and is never a feature.

### Context
- `content_id`, `client_id`
- `provider_used`, `model_used`

IDs and provenance fields support grouping, joining, auditing, or splitting; they are not model inputs.

### Excluded
- `trend_direction`: source of the label, so using it would leak the answer.
- `trend_pct`: directly describes the change used to define decline and is unavailable as a pre-decision feature in this contract.
- Any product score, action flag, raw query, URL, client name, or private identifier: not present in this public-safe starter file and not permitted as a feature.

In [5]:
candidate_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "impressions_last_30d",
    "clicks_last_30d", "sessions_last_30d", "impressions_prev_30d",
    "clicks_prev_30d", "sessions_prev_30d", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "competition_level", "content_type",
    "main_intent", "age_tier", "freshness_tier", "word_count_tier",
    "char_count_tier", "impression_tier", "position_tier",
]

context_fields = ["content_id", "client_id", "provider_used", "model_used"]
excluded_fields = ["trend_direction", "trend_pct", "is_declining_label"]

assert set(candidate_features).issubset(df.columns)
assert set(context_fields).issubset(df.columns)
assert {"trend_direction", "trend_pct"}.issubset(df.columns)
assert "is_declining_label" not in df.columns

print(f"Candidate features: {len(candidate_features)}")
print(f"Context fields: {len(context_fields)}")
print(f"Excluded/derived fields: {excluded_fields}")

Candidate features: 37
Context fields: 4
Excluded/derived fields: ['trend_direction', 'trend_pct', 'is_declining_label']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Verification Queries

The checks below verify the contract against the loaded starter file without printing raw identifiers or query-like content:

- row and column counts, plus distinct content and client counts;
- duplicate detection at the proposed `content_id` + `client_id` grain;
- missingness for the fields used in the contract, including missingness by `content_type`;
- observed windows represented by the 90-day and paired 30-day columns;
- absence of calendar date/month fields, so no calendar window is claimed.

In [7]:
# Grain and counts
key_columns = ["content_id", "client_id"]
duplicate_rows = int(df.duplicated(key_columns).sum())
label = df["trend_direction"].astype("string").str.lower().eq("down").astype(int)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Distinct content items: {df['content_id'].nunique():,}")
print(f"Distinct clients: {df['client_id'].nunique():,}")
print(f"Duplicate content/client rows: {duplicate_rows}")
print(f"Observed decline proxy rate: {label.mean():.3f}")
assert duplicate_rows == 0
assert len(df) == 30_000
assert df.shape[1] == 44

# Missingness overall and by content type
missing_columns = [
    "search_volume", "word_count", "char_count", "impressions_90d",
    "sessions_90d", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
missing_overall = df[missing_columns].isna().sum().sort_values(ascending=False)
missing_by_type = df.groupby("content_type", dropna=False)[missing_columns].apply(
    lambda frame: frame.isna().mean().round(3)
)
print("\nMissing values overall:")
print(missing_overall.to_string())
print("\nMissingness by content type:")
print(missing_by_type.to_string())

# Window and date checks
window_columns = [
    "impressions_90d", "clicks_90d", "sessions_90d",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
]
calendar_columns = [
    column for column in df.columns
    if column.lower() in {"date", "month", "report_date"}
]
print(f"\nWindow columns present: {len(window_columns)}")
print(f"Calendar date/month columns present: {calendar_columns}")
assert not calendar_columns
assert set(window_columns).issubset(df.columns)

Rows: 30,000
Columns: 44
Distinct content items: 30,000
Distinct clients: 32
Duplicate content/client rows: 0
Observed decline proxy rate: 0.542

Missing values overall:
word_count         7699
char_count         7699
search_volume      2468
scroll_rate         125
sessions_90d          0
impressions_90d       0
ctr                   0
avg_position          0
engagement_rate       0
ai_traffic_pct        0

Missingness by content type:
                    search_volume  word_count  char_count  impressions_90d  sessions_90d  ctr  avg_position  engagement_rate  scroll_rate  ai_traffic_pct
content_type                                                                                                                                             
comparison article          0.000       0.000       0.000              0.0           0.0  0.0           0.0              0.0        0.003             0.0
feedly article              1.000       0.000       0.000              0.0           0.0  0.0     

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

- The starter file is a cross-sectional snapshot, so it cannot establish page-level causal effects or calendar-time trends.
- The 90-day totals and 30-day comparison fields are aggregated windows; they do not reveal the daily path or the exact date of each observation.
- Missingness is patterned rather than automatically random, so analyses should report missingness by `content_type` and avoid treating a blank as a measured zero without justification.
- `avg_position == 0` means no position data, not a rank of zero; rate fields are percentage-point-style values, not proportions.
- Pseudonymous IDs can support grouping and client-aware validation, but they cannot explain client behavior and must not be model features.
- The decline label is an observed rule-based proxy, not proof that a page will decline in the future.

In [8]:
# Verify the important documented edge cases without exposing row-level data.
zero_position_count = int((df["avg_position"] == 0).sum())
rate_columns = ["ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct", "trend_pct"]
rate_maxima = df[rate_columns].max(numeric_only=True)

print(f"Rows with avg_position == 0 (no position data): {zero_position_count:,}")
print("Maximum observed rate-column values:")
print(rate_maxima.to_string())
print("\nContract limits: no causal or calendar-time claims; IDs remain context only.")
assert zero_position_count > 0
assert rate_maxima["scroll_rate"] > 100
assert rate_maxima["ai_traffic_pct"] >= 0

Rows with avg_position == 0 (no position data): 1,205
Maximum observed rate-column values:
ctr                  100.0
engagement_rate      100.0
scroll_rate          300.0
ai_traffic_pct       300.0
trend_pct          44900.0

Contract limits: no causal or calendar-time claims; IDs remain context only.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [x] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, URLs, or private queries anywhere.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card.